<a href="https://colab.research.google.com/github/dxda6216/Q10_Temp_vs_Activity/blob/main/Q10_temp_vs_activity_inputting_data_from_Excel_file_C1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
### This is a simple script to calculate Q10 values for activity.
### length by using SciPy Optimize non-linear least squares fit on Colab.
### Copyright (c) 2022 by dxda6216 (dxda6216 AT gmail DOT com)
###
#@title Q10 calculator (Temperature vs. Reaction Rate) {single-column: true}

import statistics
import numpy as np
import pandas as pd
from google.colab import files
from matplotlib import pyplot as plt
from scipy.optimize import curve_fit

#@markdown This is to calculate the Q10 temperature coefficient for reaction rate, enzymatic activity, frequency, etc (not for circadian period).

#@markdown For the circadian period, please use https://github.com/dxda6216/q10

#@markdown Prepare an Excel file containing multiple datasets.
#@markdown - To the 1st column (Column A), enter temperature data of the 1st dataset
#@markdown - To the 2nd column (Column B), enter activity data of the 1st dataset
#@markdown - To the 3rd column (Column C), enter temperature data of the 2nd dataset
#@markdown - To the 4th column (Column D), enter activity data of the 2nd dataset
#@markdown - ⇓
#@markdown - To the *2n-1* th column, enter temperature data of the *n* th dataset
#@markdown - To the *2n* th column, enter activity data of the *n* th dataset

#@markdown - No Index (data number, name, etc.) on the 1st row (Row 1) of the Excel sheet.

#@markdown ---

#@markdown Set the base temperature
Select_base_temperature = "30\u00B0C" #@param ["0\u00B0C", "4\u00B0C", "10\u00B0C", "15\u00B0C", "20\u00B0C", "25\u00B0C", "30\u00B0C", "35\u00B0C", "37\u00B0C", "40\u00B0C", "45\u00B0C", "50\u00B0C", "60\u00B0C", "100\u00B0C", "Minimum", "Maximum", "Average", "Median", "Set \"Base_temperature\" by slider below"]
Base_temperature = 30 # @param {type:"slider", min:0, max:100, step:0.1}

#@markdown ---

# Fixed named temperatures selectable from the dropdown above.
BASE_TEMP_MAP = {
    "0\u00B0C": 0.0, "4\u00B0C": 4.0, "10\u00B0C": 10.0, "15\u00B0C": 15.0,
    "20\u00B0C": 20.0, "25\u00B0C": 25.0, "30\u00B0C": 30.0, "35\u00B0C": 35.0,
    "37\u00B0C": 37.0, "40\u00B0C": 40.0, "45\u00B0C": 45.0, "50\u00B0C": 50.0,
    "60\u00B0C": 60.0, "100\u00B0C": 100.0,
}

# Plot appearance
MARKER_COLOR = "red"
MARKER_SIZE = 30
FIT_LINE_COLOR = "lightgreen"
X_AXIS_LABEL = "Temperature (\u00B0C)"
Y_AXIS_LABEL = "Activity"


def load_data():
    """Clear old uploads, prompt the user for a new file, and load it."""
    get_ipython().system('rm -f *.xlsx *.csv *.dat *.zip')
    uploaded = files.upload()
    data_file = next(iter(uploaded))
    return pd.read_excel(data_file, header=None, index_col=None)


def resolve_base_temperature(selection, slider_value, x):
    """Turn the dropdown selection into a numeric base temperature."""
    if selection in BASE_TEMP_MAP:
        return BASE_TEMP_MAP[selection]
    if selection == "Minimum":
        return np.min(x)
    if selection == "Maximum":
        return np.max(x)
    if selection == "Average":
        return statistics.mean(x)
    if selection == "Median":
        return statistics.median(x)
    return slider_value  # "Set \"Base_temperature\" by slider below"


def q10_model(x, rate_bt, q10, base_x):
    """Q10 rate equation: rate = q10 ** ((T - T_base) / 10) * rate_at_base."""
    return (q10 ** ((x - base_x) * 0.1)) * rate_bt


def fit_dataset(x, y, base_x):
    """Fit the Q10 model to one dataset.

    Returns the fitted parameters, their covariance matrix, R-squared,
    and the fitted function (with base_x baked in) for plotting.
    """
    # Initial guess for activity at the base temperature: use the activity
    # value(s) measured at the experimental temperature closest to base_x.
    closest_x = min(x, key=lambda v: abs(v - base_x))
    initial_rate_bt = y[x == closest_x].mean()
    p0 = [initial_rate_bt, 1.0]  # [activity at base temp, Q10]

    def func(x, rate_bt, q10):
        return q10_model(x, rate_bt, q10, base_x)

    popt, pcov = curve_fit(func, x, y, p0=p0)

    residuals = y - func(x, *popt)
    ss_residuals = np.sum(residuals ** 2)
    ss_total = np.sum((y - np.mean(y)) ** 2)
    r_squared = 1 - (ss_residuals / ss_total)

    return popt, pcov, r_squared, func


def plot_dataset(x, y, func, popt, title):
    """Scatter-plot the data with the fitted Q10 curve overlaid."""
    plt.figure(figsize=(6, 4.5))

    margin = (max(x) - min(x)) * 0.25
    fcx = np.linspace(min(x) - margin, max(x) + margin, 200)
    fcy = func(fcx, popt[0], popt[1])

    plt.scatter(x, y, s=MARKER_SIZE, color=MARKER_COLOR, label="data")
    plt.plot(fcx, fcy, "--", color=FIT_LINE_COLOR,
             label="Fit line    Q10 = %5.3f" % popt[1])
    plt.title(title)
    plt.xlabel(X_AXIS_LABEL)
    plt.ylabel(Y_AXIS_LABEL)
    plt.legend()
    plt.show()


def main():
    tadf = load_data()
    num_datasets = len(tadf.columns) // 2

    print(tadf, "\n")
    print("Number of datasets:", num_datasets, "\n")

    for i in range(num_datasets):
        temp_col, activity_col = 2 * i, 2 * i + 1
        x = tadf[temp_col].dropna().to_numpy()
        y = tadf[activity_col].dropna().to_numpy()
        dataname = f"# {i + 1}"

        print("Dataset", dataname)
        print("Temperatures: ", x)
        print("Activities: ", y)

        base_x = resolve_base_temperature(Select_base_temperature, Base_temperature, x)
        print("\n", pd.DataFrame({"Temperature": x, "Activity": y}), "\n")
        print("Base temperature =", base_x, "\u00B0C\n")

        popt, pcov, r_squared, func = fit_dataset(x, y, base_x)

        print(f"Estimated activity at {base_x}\u00B0C = {popt[0]:.3f} \u00B1 {pcov[0, 0] ** 0.5:.3f}")
        print(f"Q10 (temperature coefficient) = {popt[1]:.3f} \u00B1 {pcov[1, 1] ** 0.5:.3f}")
        print(f"R\u00B2 = {r_squared:.6f}\n")

        plot_dataset(x, y, func, popt, dataname)


main()
